# Predicting Cross-Border Music Diffusion on Spotify

## Business Context

When a song enters a national Spotify chart for the first time, marketing teams and playlist curators face an immediate question: **which other countries will this song chart in next?** Answering this within days — not weeks — enables targeted ad spend, localized playlist placement, and coordinated label rollouts across 66 global markets.

This notebook builds a **Top-5 Country Ranker**: given a song's first chart appearance, it predicts the 5 countries most likely to see the song chart within 60 days. The model is trained on 25.1M chart observations spanning 2017–2021.

**Why 60 days?** Industry research on viral music propagation shows that cross-border chart entry follows a heavy-tailed distribution: the majority of international spread occurs within the first 2–4 weeks after initial charting, with a long tail extending to ~60 days. Beyond 60 days, remaining entries are rare and largely driven by external events (e.g., TikTok virality, movie soundtracks) rather than organic diffusion patterns the model can capture. The 60-day window balances recall of genuine spread events against noise from coincidental late entries.

**Why Top-5?** Tracks that spread internationally chart in a median of 5.17 additional countries (among tracks that spread at all). Predicting k=5 countries aligns with this empirical distribution — it is large enough to capture the typical spread footprint while remaining actionable for marketing teams who need to prioritize a small set of target markets. At k=3, the theoretical recall ceiling is too low (~58%); at k=10, the list becomes less actionable and precision drops substantially.

## Technical Approach

### Why XGBoost?

We chose gradient-boosted decision trees (XGBoost) over alternative model families for several reasons:

- **Tabular data strength:** Our feature matrix consists of heterogeneous tabular features (continuous distances, categorical flags, count statistics). Gradient boosting consistently outperforms neural networks on structured tabular data (Grinsztajn et al., 2022), and our dataset lacks the sequential or spatial structure that would favor RNNs or CNNs.
- **Native learning-to-rank support:** XGBoost's `XGBRanker` with `rank:ndcg` directly optimizes the listwise ranking objective we care about. Random forests and standard neural networks require custom loss functions or surrogate objectives for ranking tasks.
- **Handling of missing values:** XGBoost natively learns optimal split directions for missing features (19.4% of cultural distance features are missing due to incomplete Hofstede data). This avoids brittle imputation strategies.
- **Interpretability:** Feature importance via gain decomposition allows us to validate that the model relies on meaningful cultural and geographic signals, not artifacts — critical for academic evaluation and business trust.
- **XGBoost vs LightGBM:** Both are competitive gradient boosting frameworks. We chose XGBoost for its mature `XGBRanker` API and broader academic adoption. LightGBM's `lambdarank` is a viable alternative but offered no clear advantage for our dataset size.

### Model Architecture

We implement a **2-stage system** with Optuna-tuned XGBoost models:

1. **Country Ranker (primary model):** An XGBRanker trained on row-level (track, target_country) pairs using the `rank:ndcg` listwise objective. For each new song, it scores all 65 candidate countries and returns the top 5. Tuned with Optuna TPE (50 trials) using temporal cross-validation.

2. **Days-to-Entry Regressor (auxiliary):** An XGBRegressor that estimates *when* the song will chart in each predicted country. Useful for operational planning (e.g., "launch the campaign in Brazil within 3 days").

We also train a **will-spread classifier** as an ablation study to evaluate whether a pre-filtering gate improves the pipeline. We conclude that it does not — the standalone ranker delivers the best recall.

### Why Learning-to-Rank over Pointwise Classification?

Our task is inherently a ranking problem: given a track, produce an ordered list of countries. Two formulations are possible:

- **Pointwise classification** (NB 08): Train a binary classifier on each (track, country) pair to predict P(entry). Rank countries by predicted probability. This ignores inter-country dependencies — the model doesn't know that predicting "Germany" affects the optimal prediction for "Austria."
- **Listwise learning-to-rank** (this notebook): XGBRanker with `rank:ndcg` optimizes the *ordering* directly, using group-aware gradients that consider all candidates for each track simultaneously.

Our experiments confirm the theoretical advantage: the ranker achieves recall@5 = 0.669 vs the classifier's 0.618 (+8.2% relative), with even larger gains in ndcg@5 (0.727 vs 0.651, +11.7%). The ranker places relevant countries higher in the list, not just somewhere in the top-5.

## Key Results

| Model | Test recall@5 | Test ndcg@5 | Test hit_rate@5 |
|-------|--------------|-------------|----------------|
| **Final Ranker (this notebook)** | **0.669** | **0.727** | **0.874** |
| Previous best (NB 09) | 0.669 | 0.725 | 0.875 |
| XGBClassifier baseline (NB 08) | 0.618 | 0.651 | 0.837 |
| Naive popularity baseline | 0.071 | 0.106 | 0.242 |

The model correctly identifies **2 out of 3 future chart countries** in its top-5 predictions, a 9.4× improvement over the popularity baseline.

In [ ]:
from pathlib import Path
import json
import os
import pickle
import tempfile
import warnings

os.environ.setdefault('MPLCONFIGDIR', tempfile.mkdtemp(prefix='matplotlib-'))

import duckdb
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import optuna
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    fbeta_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from IPython.display import display
import xgboost as xgb

optuna.logging.set_verbosity(optuna.logging.INFO)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'datasets').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. Expected a parent containing 'datasets/' and 'requirements.txt'."
    )


ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = ROOT / 'datasets' / 'processed' / 'v3_features'
MODEL_DIR = ROOT / 'artifacts' / 'models' / 'xgboost_final_pipeline'
EVAL_DIR = ROOT / 'artifacts' / 'evaluations' / 'xgboost_final_pipeline'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.parquet'
VAL_PATH = DATA_DIR / 'val.parquet'
TEST_PATH = DATA_DIR / 'test.parquet'

assert TRAIN_PATH.exists(), f'Missing training split: {TRAIN_PATH}'
assert VAL_PATH.exists(), f'Missing validation split: {VAL_PATH}'
assert TEST_PATH.exists(), f'Missing test split: {TEST_PATH}'

# --- Config ---
RANDOM_STATE = 42
TOP_K = 5

RANKER_N_TRIALS = 50
CLASSIFIER_N_TRIALS = 30
REGRESSOR_N_TRIALS = 30

TIME_BLOCKS = 5
OPTUNA_N_STARTUP_TRIALS = 10
EARLY_STOPPING_ROUNDS = 50
TUNING_N_ESTIMATORS = 2000
FINAL_N_ESTIMATOR_BUFFER = 1.10

PRIMARY_PRECISION_FLOOR = 0.20
BUSINESS_PRECISION_FLOORS = [0.20, 0.25]

# Prior-notebook artifact paths (for feature selection + comparison)
NB09_EVAL_DIR = ROOT / 'artifacts' / 'evaluations' / 'xgboost_ranker_temporal_tuned'
NB10_EVAL_DIR = ROOT / 'artifacts' / 'evaluations' / 'xgboost_will_spread_classifier'

print({
    'ranker_n_trials': RANKER_N_TRIALS,
    'classifier_n_trials': CLASSIFIER_N_TRIALS,
    'regressor_n_trials': REGRESSOR_N_TRIALS,
    'time_blocks': TIME_BLOCKS,
    'top_k': TOP_K,
})

## 1. Setup and Helper Functions

All shared utilities for data loading, temporal cross-validation, ranking metrics, and model fitting.

### Temporal Train/Validation/Test Split

We split data by the track's first chart date: **train ≤ 2019, validation = 2020, test = 2021**. This strict temporal split is essential because:

- **Preventing data leakage:** Random k-fold CV would allow the model to train on 2021 tracks and predict 2019 tracks — an impossible scenario in production where we only have historical data. Music trends, artist popularity, and platform dynamics evolve over time; a model must prove it generalizes *forward*.
- **Why full calendar years?** Music diffusion is seasonal (holiday releases, summer hits, award season). Year-level splits ensure each split contains a full seasonal cycle, avoiding bias from partial-year effects. Finer splits (e.g., monthly) would introduce boundary artifacts where a track's 60-day prediction horizon crosses the split boundary.
- **Temporal CV within training:** During hyperparameter tuning, we further split the training set (≤2019) into 3 expanding-window folds using `make_temporal_folds()` (5 time blocks → 3 folds). Each fold trains on earlier blocks and validates on the next, maintaining temporal ordering even within the training set.

### Evaluation Metrics

We evaluate the ranker using standard information retrieval metrics at cutoff k=5:

- **recall@k = |relevant ∩ predicted@k| / |relevant|** — Fraction of actually-spreading countries captured in the top-k predictions. Our primary metric: directly measures how many market opportunities the model identifies.
- **ndcg@k (Normalized Discounted Cumulative Gain)** — Measures ranking quality by penalizing relevant items placed lower in the list. Defined as DCG@k / IDCG@k, where DCG@k = Σᵢ (relᵢ / log₂(i+1)) for positions i=1..k. A model that places the most important countries first scores higher.
- **MAP@k (Mean Average Precision)** — Average of precision@j for each position j where a relevant country appears, averaged across all tracks. Rewards models that cluster relevant countries at the top of the list.
- **hit_rate@k** — Fraction of tracks where at least one predicted country is correct. Measures broad reliability across the track population.

For the classifier ablation, we use **average precision (AP)**, **Brier score** (calibration quality), and **F₂ score** (recall-weighted F-measure, reflecting the higher cost of missing a spreading track vs. false alarms).

In [ ]:
con = duckdb.connect()

FEATURE_EXCLUDE = ['track_id', 'observation_time', 'target_country', 'did_enter_within_60d', 'days_to_entry']
TARGET_SPECIFIC_COLS = [
    'artist_prior_success_in_target',
    'target_population',
    'target_avg_daily_streams',
    'target_new_entry_rate_30d',
    'target_continent_africa',
    'target_continent_asia',
    'target_continent_europe',
    'target_continent_north_america',
    'target_continent_oceania',
    'target_continent_south_america',
    'cultural_dist_min',
    'cultural_dist_missing',
    'same_language_flag',
    'same_continent_flag',
    'neighbor_entered_count',
]
EXCLUDE_FROM_TRACK_LEVEL = {'target_country', 'did_enter_within_60d', 'days_to_entry'}


# ── Data loading ──────────────────────────────────────────────────────────────

def load_row_level_split(path: Path, max_tracks: int | None = None) -> pd.DataFrame:
    parquet_path = path.as_posix()
    if max_tracks is None:
        query = f"SELECT * FROM read_parquet('{parquet_path}')"
    else:
        query = f"""
            WITH sampled_tracks AS (
                SELECT track_id
                FROM read_parquet('{parquet_path}')
                GROUP BY track_id
                ORDER BY hash(track_id)
                LIMIT {int(max_tracks)}
            )
            SELECT d.*
            FROM read_parquet('{parquet_path}') d
            JOIN sampled_tracks st USING (track_id)
        """
    df = con.execute(query).fetchdf()
    df['observation_time'] = pd.to_datetime(df['observation_time'])
    return df


def load_track_level_split(path: Path, max_tracks: int | None = None) -> pd.DataFrame:
    parquet_path = path.as_posix()
    schema_df = con.execute(f"SELECT * FROM read_parquet('{parquet_path}') LIMIT 0").fetchdf()
    raw_cols = list(schema_df.columns)
    constant_cols = [c for c in raw_cols if c not in EXCLUDE_FROM_TRACK_LEVEL and c not in TARGET_SPECIFIC_COLS]
    rank_cols = [c for c in raw_cols if c.startswith('rank_')]

    source_expr = f"read_parquet('{parquet_path}')"
    if max_tracks is None:
        source_query = source_expr
    else:
        source_query = f"""
            (
                WITH sampled_tracks AS (
                    SELECT track_id
                    FROM {source_expr}
                    GROUP BY track_id
                    ORDER BY hash(track_id)
                    LIMIT {int(max_tracks)}
                )
                SELECT d.*
                FROM {source_expr} d
                JOIN sampled_tracks st USING (track_id)
            )
        """

    select_parts = [
        'track_id',
        'MAX(CAST(did_enter_within_60d AS INTEGER)) AS will_spread',
        'MIN(CASE WHEN did_enter_within_60d = 1 THEN days_to_entry END) AS min_days_to_spread',
        'COUNT(*) AS candidate_count',
    ]
    select_parts.extend([f'ANY_VALUE({col}) AS {col}' for col in constant_cols if col != 'track_id'])
    for col in TARGET_SPECIFIC_COLS:
        select_parts.append(f'AVG({col}) AS {col}_mean')
        select_parts.append(f'MAX({col}) AS {col}_max')

    query = f"""
        SELECT
            {', '.join(select_parts)}
        FROM {source_query}
        GROUP BY track_id
        ORDER BY track_id
    """
    df = con.execute(query).fetchdf()
    df['observation_time'] = pd.to_datetime(df['observation_time'])
    df['origin_country_count_at_obs'] = (df[rank_cols].fillna(0) > 0).sum(axis=1)
    return df


def make_feature_matrix(df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series) -> pd.DataFrame:
    return df[feature_cols].copy().fillna(fill_values)


# ── Temporal CV ───────────────────────────────────────────────────────────────

def make_temporal_folds(df: pd.DataFrame, time_blocks: int = 5) -> list[dict]:
    track_times = df[['track_id', 'observation_time']].drop_duplicates().sort_values(
        ['observation_time', 'track_id']
    ).reset_index(drop=True)
    track_times['time_block'] = pd.qcut(track_times.index, q=time_blocks, labels=False, duplicates='drop')
    folds = []
    unique_blocks = sorted(track_times['time_block'].dropna().astype(int).unique())
    for fold_idx, block in enumerate(unique_blocks[2:], start=1):
        train_track_ids = track_times.loc[track_times['time_block'] < block, 'track_id'].tolist()
        val_track_ids = track_times.loc[track_times['time_block'] == block, 'track_id'].tolist()
        if not train_track_ids or not val_track_ids:
            continue
        train_times = track_times.loc[track_times['time_block'] < block, 'observation_time']
        val_times = track_times.loc[track_times['time_block'] == block, 'observation_time']
        folds.append({
            'fold': fold_idx,
            'train_track_ids': train_track_ids,
            'val_track_ids': val_track_ids,
            'train_tracks': len(train_track_ids),
            'val_tracks': len(val_track_ids),
            'train_start': str(train_times.min().date()),
            'train_end': str(train_times.max().date()),
            'val_start': str(val_times.min().date()),
            'val_end': str(val_times.max().date()),
        })
    return folds


# ── Metrics ───────────────────────────────────────────────────────────────────

def safe_roc_auc(y_true, y_score) -> float | None:
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return None
    return float(roc_auc_score(y_true, y_score))


def safe_average_precision(y_true, y_score) -> float | None:
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return None
    return float(average_precision_score(y_true, y_score))


def binary_metrics(y_true, y_score) -> dict:
    clipped = np.clip(y_score, 1e-6, 1 - 1e-6)
    return {
        'roc_auc': safe_roc_auc(y_true, y_score),
        'average_precision': safe_average_precision(y_true, y_score),
        'log_loss': float(log_loss(y_true, clipped, labels=[0, 1])),
    }


def build_threshold_table(y_true, y_prob, beta: float = 2.0) -> pd.DataFrame:
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    rows = []
    beta_sq = beta ** 2
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        denom = p + r
        f1 = 0.0 if denom == 0 else 2 * p * r / denom
        fbeta = 0.0 if (beta_sq * p + r) == 0 else (1 + beta_sq) * p * r / (beta_sq * p + r)
        predicted_positive_rate = float((y_prob >= t).mean())
        rows.append({
            'threshold': float(t),
            'precision': float(p),
            'recall': float(r),
            'f1': float(f1),
            f'f{beta}': float(fbeta),
            'predicted_positive_rate': predicted_positive_rate,
            'flagged_per_1000_tracks': predicted_positive_rate * 1000.0,
        })
    return pd.DataFrame(rows).sort_values(['threshold']).reset_index(drop=True)


def choose_recall_threshold(y_true, y_prob, precision_floor: float, beta: float = 2.0):
    threshold_df = build_threshold_table(y_true, y_prob, beta=beta)
    eligible = threshold_df[threshold_df['precision'] >= precision_floor].copy()
    if not eligible.empty:
        selected = eligible.sort_values(['recall', 'precision', 'threshold'], ascending=[False, False, True]).iloc[0]
        reason = f'highest recall with precision >= {precision_floor:.2f}'
    else:
        score_col = f'f{beta}'
        selected = threshold_df.sort_values([score_col, 'recall', 'precision'], ascending=[False, False, False]).iloc[0]
        reason = f'fallback to best {score_col} because no threshold met precision floor {precision_floor:.2f}'
    return float(selected['threshold']), threshold_df, reason


def binary_summary(y_true, y_prob, threshold: float, beta: float = 2.0) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    clipped = np.clip(y_prob, 1e-6, 1 - 1e-6)
    return {
        'roc_auc': safe_roc_auc(y_true, y_prob),
        'average_precision': safe_average_precision(y_true, y_prob),
        'log_loss': float(log_loss(y_true, clipped, labels=[0, 1])),
        'brier_score': float(brier_score_loss(y_true, clipped)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        f'f{beta}': float(fbeta_score(y_true, y_pred, beta=beta, zero_division=0)),
        'positive_rate': float(np.mean(y_true)),
        'predicted_positive_rate': float(np.mean(y_pred)),
        'flagged_per_1000_tracks': float(np.mean(y_pred) * 1000.0),
    }


def build_business_threshold_summary(y_true, y_prob, floors: list[float], beta: float = 2.0):
    rows = []
    threshold_tables = []
    threshold_map = {}
    for floor in floors:
        threshold, threshold_df, reason = choose_recall_threshold(y_true, y_prob, precision_floor=floor, beta=beta)
        metrics = binary_summary(y_true, y_prob, threshold=threshold, beta=beta)
        rows.append({
            'precision_floor': float(floor),
            'threshold': float(threshold),
            'selection_reason': reason,
            **metrics,
        })
        threshold_map[float(floor)] = float(threshold)
        threshold_df = threshold_df.copy()
        threshold_df['precision_floor'] = float(floor)
        threshold_tables.append(threshold_df)
    return pd.DataFrame(rows), threshold_map, pd.concat(threshold_tables, ignore_index=True)


def compute_candidate_stats(scored_df: pd.DataFrame) -> dict:
    per_track = scored_df.groupby('track_id').agg(
        candidates=('target_country', 'size'),
        positives=('did_enter_within_60d', 'sum'),
    )
    positive_mask = per_track['positives'] > 0
    return {
        'tracks': int(per_track.shape[0]),
        'positive_tracks': int(positive_mask.sum()),
        'avg_candidates_per_track': float(per_track['candidates'].mean()),
        'avg_future_countries_per_track': float(per_track['positives'].mean()),
        'avg_future_countries_per_positive_track': float(per_track.loc[positive_mask, 'positives'].mean()) if positive_mask.any() else None,
    }


def ranking_metrics(scored_df: pd.DataFrame, k: int = 5) -> tuple[dict, pd.DataFrame]:
    rows = []
    for track_id, group in scored_df.groupby('track_id', sort=False):
        group = group.sort_values(['score', 'tie_break'], ascending=[False, False]).reset_index(drop=True)
        labels = group['did_enter_within_60d'].to_numpy(dtype=int)
        positives = int(labels.sum())
        top = group.head(k)
        top_labels = top['did_enter_within_60d'].to_numpy(dtype=int)
        hits = int(top_labels.sum())

        precision = hits / k
        recall = hits / positives if positives else np.nan
        hit_rate = float(hits > 0) if positives else np.nan

        discounts = np.log2(np.arange(2, len(top_labels) + 2))
        dcg = float(((2 ** top_labels - 1) / discounts).sum())
        ideal = np.sort(labels)[::-1][:len(top_labels)]
        idcg = float(((2 ** ideal - 1) / np.log2(np.arange(2, len(ideal) + 2))).sum())
        ndcg = dcg / idcg if idcg > 0 else np.nan

        ap_accum = 0.0
        running_hits = 0
        for rank, rel in enumerate(top_labels, start=1):
            if rel:
                running_hits += 1
                ap_accum += running_hits / rank
        map_k = ap_accum / min(positives, k) if positives else np.nan

        rows.append({
            'track_id': track_id,
            'positives': positives,
            'top_k_hits': hits,
            f'precision@{k}': precision,
            f'recall@{k}': recall,
            f'hit_rate@{k}': hit_rate,
            f'ndcg@{k}': ndcg,
            f'map@{k}': map_k,
        })

    metric_df = pd.DataFrame(rows)
    positive_mask = metric_df['positives'] > 0
    summary = {
        'tracks': int(metric_df.shape[0]),
        'positive_tracks': int(positive_mask.sum()),
        f'precision@{k}': float(metric_df[f'precision@{k}'].mean()),
        f'recall@{k}': float(metric_df.loc[positive_mask, f'recall@{k}'].mean()) if positive_mask.any() else None,
        f'hit_rate@{k}': float(metric_df.loc[positive_mask, f'hit_rate@{k}'].mean()) if positive_mask.any() else None,
        f'ndcg@{k}': float(metric_df.loc[positive_mask, f'ndcg@{k}'].mean()) if positive_mask.any() else None,
        f'map@{k}': float(metric_df.loc[positive_mask, f'map@{k}'].mean()) if positive_mask.any() else None,
        'mean_future_countries_per_track': float(metric_df['positives'].mean()),
    }
    return summary, metric_df


def evaluate_ranked_candidates(scored_df: pd.DataFrame, k: int = 5) -> tuple[dict, pd.DataFrame]:
    candidate_stats = compute_candidate_stats(scored_df)
    binary = binary_metrics(scored_df['did_enter_within_60d'], scored_df['score'].to_numpy())
    ranking_all, track_metrics = ranking_metrics(scored_df, k=k)
    positive_track_metrics = track_metrics[track_metrics['positives'] > 0].copy()
    positive_summary = {
        'tracks': int(positive_track_metrics.shape[0]),
        'avg_future_countries_per_positive_track': float(positive_track_metrics['positives'].mean()) if not positive_track_metrics.empty else None,
        f'avg_top_{k}_hits_per_positive_track': float(positive_track_metrics['top_k_hits'].mean()) if not positive_track_metrics.empty else None,
        f'recall@{k}': float(positive_track_metrics[f'recall@{k}'].mean()) if not positive_track_metrics.empty else None,
        f'hit_rate@{k}': float(positive_track_metrics[f'hit_rate@{k}'].mean()) if not positive_track_metrics.empty else None,
        f'ndcg@{k}': float(positive_track_metrics[f'ndcg@{k}'].mean()) if not positive_track_metrics.empty else None,
        f'map@{k}': float(positive_track_metrics[f'map@{k}'].mean()) if not positive_track_metrics.empty else None,
    }
    return {
        'binary': binary,
        'candidate_stats': candidate_stats,
        'ranking_all_tracks': ranking_all,
        'ranking_positive_tracks': positive_summary,
    }, track_metrics


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    abs_err = np.abs(y_true - y_pred)
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'median_ae': float(median_absolute_error(y_true, y_pred)),
        'pct_within_3_days': float(np.mean(abs_err <= 3.0)),
        'pct_within_7_days': float(np.mean(abs_err <= 7.0)),
    }


# ── Scoring helpers ──────────────────────────────────────────────────────────

def normalize_scores(values: pd.Series) -> pd.Series:
    value_min = float(values.min())
    value_max = float(values.max())
    if value_max > value_min:
        return (values - value_min) / (value_max - value_min)
    return pd.Series(np.full(len(values), 0.5), index=values.index)


def score_naive_popularity(df: pd.DataFrame) -> pd.DataFrame:
    scored = df[['track_id', 'observation_time', 'target_country', 'did_enter_within_60d', 'days_to_entry', 'target_avg_daily_streams', 'target_new_entry_rate_30d']].copy()
    primary = scored['target_avg_daily_streams'].fillna(0.0)
    tie_break = scored['target_new_entry_rate_30d'].fillna(0.0)
    raw_score = primary + tie_break * 1e-6
    scored['score'] = normalize_scores(raw_score)
    scored['tie_break'] = tie_break
    scored['model_name'] = 'naive_popularity_baseline'
    return scored


def feature_category(name: str) -> str:
    if name.startswith('rank_') or name in {'track_in_viral50_at_obs', 'candidate_count', 'origin_country_count_at_obs'}:
        return 'current_footprint'
    if name.startswith('artist_') or name == 'multi_artist_flag':
        return 'artist_history'
    if name.startswith('target_'):
        return 'target_country_priors'
    if name in {'cultural_dist_min', 'cultural_dist_missing', 'same_language_flag', 'same_continent_flag', 'neighbor_entered_count'}:
        return 'origin_target_relationship'
    if name.endswith('_mean') or name.endswith('_max'):
        return 'aggregates'
    if name.startswith('af_') or name in {'duration_ms', 'explicit', 'days_since_release', 'is_friday_release'}:
        return 'audio_track_metadata'
    if name.startswith('observation_'):
        return 'temporal'
    return 'other'


# ── Ranker helpers ────────────────────────────────────────────────────────────

def prepare_ranker_inputs(df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series):
    ordered = df.sort_values(['track_id', 'target_country']).reset_index(drop=True)
    X = make_feature_matrix(ordered, feature_cols, fill_values)
    y = ordered['did_enter_within_60d'].astype(float).to_numpy()
    group = ordered.groupby('track_id', sort=False).size().to_numpy()
    return ordered, X, y, group


def make_ranker(params: dict, n_estimators: int = TUNING_N_ESTIMATORS, early_stopping_rounds: int | None = EARLY_STOPPING_ROUNDS):
    init_kwargs = {
        'objective': 'rank:ndcg',
        'eval_metric': f'ndcg@{TOP_K}',
        'tree_method': 'hist',
        'n_estimators': n_estimators,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        **params,
    }
    if early_stopping_rounds is not None:
        init_kwargs['early_stopping_rounds'] = early_stopping_rounds
    return xgb.XGBRanker(**init_kwargs)


def score_ranker(model, df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, model_name: str) -> pd.DataFrame:
    ordered = df.sort_values(['track_id', 'target_country']).reset_index(drop=True)
    X = make_feature_matrix(ordered, feature_cols, fill_values)
    raw_scores = pd.Series(model.predict(X), index=ordered.index)
    scored = ordered[['track_id', 'observation_time', 'target_country', 'did_enter_within_60d', 'days_to_entry', 'target_avg_daily_streams', 'target_new_entry_rate_30d']].copy()
    scored['score'] = normalize_scores(raw_scores)
    scored['raw_score'] = raw_scores.to_numpy()
    scored['tie_break'] = scored['target_new_entry_rate_30d'].fillna(0.0)
    scored['model_name'] = model_name
    return scored


def fit_ranker_with_validation(train_part, val_part, feature_cols, fill_values, params):
    _, X_train, y_train, group_train = prepare_ranker_inputs(train_part, feature_cols, fill_values)
    _, X_val, y_val, group_val = prepare_ranker_inputs(val_part, feature_cols, fill_values)
    model = make_ranker(params)
    model.fit(X_train, y_train, group=group_train, eval_set=[(X_val, y_val)], eval_group=[group_val], verbose=False)
    return model


def fit_final_ranker(train_part, feature_cols, fill_values, params, n_estimators):
    _, X_train, y_train, group_train = prepare_ranker_inputs(train_part, feature_cols, fill_values)
    model = make_ranker(params, n_estimators=n_estimators, early_stopping_rounds=None)
    model.fit(X_train, y_train, group=group_train, verbose=False)
    return model


# ── Regressor helpers ─────────────────────────────────────────────────────────

def transform_target(y, target_transform: str) -> np.ndarray:
    y_arr = np.asarray(y, dtype=float)
    if target_transform == 'log1p':
        return np.log1p(y_arr)
    if target_transform == 'sqrt':
        return np.sqrt(y_arr)
    return y_arr


def inverse_transform_target(y: np.ndarray, target_transform: str) -> np.ndarray:
    y_arr = np.asarray(y, dtype=float)
    if target_transform == 'log1p':
        return np.expm1(y_arr)
    if target_transform == 'sqrt':
        return np.square(y_arr)
    return y_arr


def make_regressor(params: dict, n_estimators: int = 800, early_stopping_rounds: int | None = 30):
    init_kwargs = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'tree_method': 'hist',
        'n_estimators': n_estimators,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        **params,
    }
    if early_stopping_rounds is not None:
        init_kwargs['early_stopping_rounds'] = early_stopping_rounds
    return xgb.XGBRegressor(**init_kwargs)


def score_regressor(model, df, feature_cols, fill_values, target_transform='identity'):
    preds = model.predict(make_feature_matrix(df, feature_cols, fill_values))
    preds = inverse_transform_target(preds, target_transform)
    scored = df[['track_id', 'observation_time', 'target_country', 'did_enter_within_60d', 'days_to_entry']].copy()
    scored['predicted_days_to_entry'] = np.clip(preds, 1.0, 60.0)
    return scored


# ── Pipeline integration helpers ──────────────────────────────────────────────

def add_stage1_to_row_predictions(row_scored_df, stage1_scored_df, threshold_map):
    merged = row_scored_df.merge(
        stage1_scored_df[['track_id', 'score']].rename(columns={'score': 'spread_score'}),
        on='track_id', how='left',
    )
    for floor, threshold in threshold_map.items():
        col = f'spread_decision_floor_{float(floor):.2f}'.replace('.', '_')
        merged[col] = (merged['spread_score'] >= float(threshold)).astype(int)
    return merged


def add_regression_predictions(row_scored_df, reg_scored_df):
    return row_scored_df.merge(
        reg_scored_df[['track_id', 'target_country', 'predicted_days_to_entry']],
        on=['track_id', 'target_country'], how='left',
    )


def add_predicted_rank(scored_df):
    ranked = scored_df.sort_values(['track_id', 'score', 'target_new_entry_rate_30d'], ascending=[True, False, False]).copy()
    ranked['predicted_rank'] = ranked.groupby('track_id').cumcount().add(1).astype(int)
    return ranked


def evaluate_gated_pipeline(scored_df, gate_col, k=5):
    rows = []
    hit_rows = []
    topk_rows = []
    for track_id, group in scored_df.groupby('track_id', sort=False):
        group = group.sort_values(['score', 'target_new_entry_rate_30d'], ascending=[False, False]).reset_index(drop=True)
        labels = group['did_enter_within_60d'].to_numpy(dtype=int)
        positives = int(labels.sum())
        gated = bool(group[gate_col].iloc[0]) if not group.empty else False
        top = group.head(k).copy() if gated else group.head(0).copy()
        top_labels = top['did_enter_within_60d'].to_numpy(dtype=int)
        hits = int(top_labels.sum())

        precision = hits / k
        recall = hits / positives if positives else np.nan
        hit_rate = float(hits > 0) if positives else np.nan

        discounts = np.log2(np.arange(2, len(top_labels) + 2)) if len(top_labels) else np.array([])
        dcg = float(((2 ** top_labels - 1) / discounts).sum()) if len(top_labels) else 0.0
        ideal = np.sort(labels)[::-1][:k]
        idcg = float(((2 ** ideal - 1) / np.log2(np.arange(2, len(ideal) + 2))).sum()) if positives else 0.0
        ndcg = dcg / idcg if idcg > 0 else np.nan

        ap_accum = 0.0
        running_hits = 0
        for rank, rel in enumerate(top_labels, start=1):
            if rel:
                running_hits += 1
                ap_accum += running_hits / rank
        map_k = ap_accum / min(positives, k) if positives else np.nan

        rows.append({
            'track_id': track_id, 'positives': positives, 'stage1_flagged': int(gated),
            'top_k_hits': hits, f'precision@{k}': precision, f'recall@{k}': recall,
            f'hit_rate@{k}': hit_rate, f'ndcg@{k}': ndcg, f'map@{k}': map_k,
        })
        if not top.empty:
            topk_rows.append(top)
            hits_df = top[top['did_enter_within_60d'] == 1].copy()
            if not hits_df.empty:
                hit_rows.append(hits_df)

    metric_df = pd.DataFrame(rows)
    positive_mask = metric_df['positives'] > 0
    summary = {
        'tracks': int(metric_df.shape[0]),
        'positive_tracks': int(positive_mask.sum()),
        'flagged_tracks': int(metric_df['stage1_flagged'].sum()),
        f'precision@{k}': float(metric_df[f'precision@{k}'].mean()),
        f'recall@{k}': float(metric_df.loc[positive_mask, f'recall@{k}'].mean()) if positive_mask.any() else None,
        f'hit_rate@{k}': float(metric_df.loc[positive_mask, f'hit_rate@{k}'].mean()) if positive_mask.any() else None,
        f'ndcg@{k}': float(metric_df.loc[positive_mask, f'ndcg@{k}'].mean()) if positive_mask.any() else None,
        f'map@{k}': float(metric_df.loc[positive_mask, f'map@{k}'].mean()) if positive_mask.any() else None,
    }
    topk_df = pd.concat(topk_rows, ignore_index=True) if topk_rows else scored_df.head(0).copy()
    hit_df = pd.concat(hit_rows, ignore_index=True) if hit_rows else scored_df.head(0).copy()
    if not hit_df.empty:
        day_metrics = regression_metrics(hit_df['days_to_entry'].astype(float).to_numpy(), hit_df['predicted_days_to_entry'].astype(float).to_numpy())
        day_metrics['evaluated_rows'] = int(len(hit_df))
    else:
        day_metrics = {'mae': None, 'rmse': None, 'median_ae': None, 'pct_within_3_days': None, 'pct_within_7_days': None, 'evaluated_rows': 0}
    return summary, metric_df, day_metrics, topk_df


print('All helper functions defined.')


def fit_regressor_with_validation(train_pos, val_pos, feature_cols, fill_values, params, target_transform='identity'):
    X_train = make_feature_matrix(train_pos, feature_cols, fill_values)
    y_train = transform_target(train_pos['days_to_entry'].astype(float), target_transform)
    X_val = make_feature_matrix(val_pos, feature_cols, fill_values)
    y_val = transform_target(val_pos['days_to_entry'].astype(float), target_transform)
    model = make_regressor(params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model


def fit_final_regressor(train_pos, feature_cols, fill_values, params, n_estimators, target_transform='identity'):
    X_train = make_feature_matrix(train_pos, feature_cols, fill_values)
    y_train = transform_target(train_pos['days_to_entry'].astype(float), target_transform)
    model = make_regressor(params, n_estimators=n_estimators, early_stopping_rounds=None)
    model.fit(X_train, y_train, verbose=False)
    return model


In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────

row_train_df = load_row_level_split(TRAIN_PATH)
row_val_df = load_row_level_split(VAL_PATH)
row_test_df = load_row_level_split(TEST_PATH)

track_train_df = load_track_level_split(TRAIN_PATH)
track_val_df = load_track_level_split(VAL_PATH)
track_test_df = load_track_level_split(TEST_PATH)

# Row-level features (one row per track x target_country pair)
row_feature_cols = [col for col in row_train_df.columns if col not in FEATURE_EXCLUDE]
row_fill_values_train = row_train_df[row_feature_cols].median(numeric_only=True)
row_fill_values_final = pd.concat([row_train_df, row_val_df], ignore_index=True)[row_feature_cols].median(numeric_only=True)

# Track-level features (one row per track, for the classifier ablation)
track_feature_exclude = ['track_id', 'observation_time', 'will_spread', 'min_days_to_spread']
track_feature_cols = [col for col in track_train_df.columns if col not in track_feature_exclude]
track_fill_values = track_train_df[track_feature_cols].median(numeric_only=True)

# Temporal CV folds: 5 time blocks → 3 expanding-window folds (shared across all stages)
time_folds = make_temporal_folds(row_train_df, time_blocks=TIME_BLOCKS)

# Positive rows for the days-to-entry regressor
positive_train_rows = row_train_df[(row_train_df['did_enter_within_60d'] == 1) & row_train_df['days_to_entry'].notna()].copy()
positive_val_rows = row_val_df[(row_val_df['did_enter_within_60d'] == 1) & row_val_df['days_to_entry'].notna()].copy()
positive_test_rows = row_test_df[(row_test_df['did_enter_within_60d'] == 1) & row_test_df['days_to_entry'].notna()].copy()

# ── Summary ───────────────────────────────────────────────────────────────────
train_tracks = row_train_df['track_id'].nunique()
val_tracks = row_val_df['track_id'].nunique()
test_tracks = row_test_df['track_id'].nunique()
spread_rate = track_train_df['will_spread'].mean() * 100

print('=== Dataset Overview ===')
print(f'Temporal split: train (2017-2019) → val (2020) → test (2021)')
print(f'  Train: {len(row_train_df):>10,} rows | {train_tracks:>6,} tracks')
print(f'  Val:   {len(row_val_df):>10,} rows | {val_tracks:>6,} tracks')
print(f'  Test:  {len(row_test_df):>10,} rows | {test_tracks:>6,} tracks')
print(f'  Features: {len(row_feature_cols)} per (track, country) pair')
print()
print(f'Only {spread_rate:.1f}% of tracks spread to at least one new country within 60 days.')
print(f'Spreading tracks enter {track_train_df.loc[track_train_df["will_spread"]==1, "candidate_count"].mean():.0f} candidate markets on average.')
print()
print('Temporal CV folds (training data only):')
display(pd.DataFrame([{k: v for k, v in f.items() if k not in ('train_track_ids', 'val_track_ids')} for f in time_folds]))

## 2. Feature Selection

Our feature matrix contains ~102 features per (track, target_country) pair, covering cultural distance, geographic proximity, artist history, chart momentum, and temporal patterns. Not all features contribute equally — prior experiments revealed that many small-country rank columns and niche cultural indicators carry zero predictive gain.

We apply a **conservative pruning strategy**: remove only features where the trained model assigned exactly zero importance (gain = 0). This eliminates noise dimensions without risking information loss, and reduces training time for hyperparameter optimization.

In [ ]:
# ── Row-level feature pruning (Stage 2 ranker) ───────────────────────────────
nb09_importance_path = NB09_EVAL_DIR / 'feature_importance.parquet'
assert nb09_importance_path.exists(), f'Missing NB09 importance: {nb09_importance_path}'

nb09_importance = con.execute(f"SELECT * FROM read_parquet('{nb09_importance_path.as_posix()}')").fetchdf()
zero_gain_row = nb09_importance[nb09_importance['gain'] == 0.0]['feature'].tolist()

pruned_row_feature_cols = [col for col in row_feature_cols if col not in zero_gain_row]
print(f'Row-level features: {len(row_feature_cols)} → {len(pruned_row_feature_cols)} (dropped {len(zero_gain_row)} zero-gain)')
print(f'Dropped: {zero_gain_row}')
print()

# ── Track-level feature pruning (Stage 1 classifier) ─────────────────────────
nb10_importance_path = NB10_EVAL_DIR / 'feature_importance.parquet'
assert nb10_importance_path.exists(), f'Missing NB10 importance: {nb10_importance_path}'

nb10_importance = con.execute(f"SELECT * FROM read_parquet('{nb10_importance_path.as_posix()}')").fetchdf()
zero_gain_track = nb10_importance[nb10_importance['gain'] == 0.0]['feature'].tolist()

pruned_track_feature_cols = [col for col in track_feature_cols if col not in zero_gain_track]
print(f'Track-level features: {len(track_feature_cols)} → {len(pruned_track_feature_cols)} (dropped {len(zero_gain_track)} zero-gain)')
print(f'Dropped: {zero_gain_track}')

# Update fill values to match pruned feature sets
pruned_row_fill_values_train = row_fill_values_train[pruned_row_feature_cols]
pruned_row_fill_values_final = row_fill_values_final[pruned_row_feature_cols]
pruned_track_fill_values = track_fill_values[pruned_track_feature_cols]

**Interpretation:** The pruned features are predominantly small-market rank columns (e.g., `rank_Luxembourg`, `rank_Latvia`) and niche cultural indicators that the model found uninformative. Removing them reduces noise and speeds up Optuna's search without sacrificing predictive power. The remaining features capture the most meaningful signals: major-market chart positions, cultural/geographic proximity, artist history, and temporal momentum.

## 3. Primary Model — XGBRanker for Country Prediction

The core of our system is a **learning-to-rank model** (XGBRanker with `rank:ndcg` objective) that directly optimizes the ordering of candidate countries for each track. This is the right formulation because Spotify's business decision is *which* markets to prioritize — the model must produce a ranked list, not just binary predictions.

**Optimization approach:**
- **Optuna TPE sampler** (Tree-structured Parzen Estimator) for Bayesian hyperparameter search. TPE models P(hyperparameters | good scores) and P(hyperparameters | bad scores) as separate kernel density estimates, then samples candidates that maximize the ratio. This is provably more sample-efficient than grid search (which scales exponentially with dimensionality) or random search (which ignores prior evaluations). With 8 hyperparameters, 50 TPE trials explore the space more effectively than hundreds of random trials (Bergstra et al., 2011).
- **MedianPruner** to early-stop unpromising trials after the first CV fold. A trial is pruned if its intermediate result is worse than the median of completed trials at the same step. We set `n_startup_trials=10` to ensure sufficient exploration before pruning begins — too few startup trials risk discarding good configurations that happen to perform poorly on the first fold.
- **Temporal cross-validation** (3 expanding-window folds) to ensure the model generalizes to future time periods, not just random holdouts. Each fold trains on earlier data and validates on the subsequent time block.
- **50 trials** exploring: learning rate [0.01–0.15], max depth [4–12], min child weight [1–100], subsample [0.6–1.0], colsample [0.5–1.0], gamma [0–5], L1/L2 regularization. The search space was informed by prior experiments and XGBoost documentation recommendations.

In [ ]:
# ── Optuna objective for Stage 2 ranker ────────────────────────────────────────

stage2_trial_records = []
stage2_fold_records = []


def ranker_objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 100.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
    }

    fold_ndcgs = []
    fold_summaries = []
    for fold_idx, fold in enumerate(time_folds):
        fold_train = row_train_df[row_train_df['track_id'].isin(fold['train_track_ids'])].copy()
        fold_val = row_train_df[row_train_df['track_id'].isin(fold['val_track_ids'])].copy()
        fold_fill = fold_train[pruned_row_feature_cols].median(numeric_only=True)

        _, X_train, y_train, group_train = prepare_ranker_inputs(fold_train, pruned_row_feature_cols, fold_fill)
        _, X_val, y_val, group_val = prepare_ranker_inputs(fold_val, pruned_row_feature_cols, fold_fill)

        model = make_ranker(params)
        model.fit(X_train, y_train, group=group_train,
                  eval_set=[(X_val, y_val)], eval_group=[group_val], verbose=False)

        scored_val = score_ranker(model, fold_val, pruned_row_feature_cols, fold_fill, 'xgb_ranker_fold')
        summary, _ = evaluate_ranked_candidates(scored_val, k=TOP_K)
        best_iter = getattr(model, 'best_iteration', None)
        resolved_rounds = int(best_iter + 1) if best_iter is not None else TUNING_N_ESTIMATORS

        ndcg = summary['ranking_all_tracks'][f'ndcg@{TOP_K}']
        fold_ndcgs.append(ndcg)

        fold_record = {
            'trial_label': f'optuna_{trial.number:03d}',
            'fold': fold['fold'],
            'best_iteration': resolved_rounds,
            f'ndcg@{TOP_K}': ndcg,
            f'recall@{TOP_K}': summary['ranking_all_tracks'][f'recall@{TOP_K}'],
            f'map@{TOP_K}': summary['ranking_all_tracks'][f'map@{TOP_K}'],
            'roc_auc': summary['binary']['roc_auc'],
        }
        fold_summaries.append(fold_record)
        stage2_fold_records.append(fold_record | params)

        # Report intermediate value for pruning
        trial.report(np.mean(fold_ndcgs), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    fold_df = pd.DataFrame(fold_summaries)
    record = {
        'trial_label': f'optuna_{trial.number:03d}',
        'mean_best_iteration': float(fold_df['best_iteration'].mean()),
        f'mean_ndcg@{TOP_K}': float(fold_df[f'ndcg@{TOP_K}'].mean()),
        f'mean_recall@{TOP_K}': float(fold_df[f'recall@{TOP_K}'].mean()),
        f'mean_map@{TOP_K}': float(fold_df[f'map@{TOP_K}'].mean()),
        'mean_roc_auc': float(fold_df['roc_auc'].mean()),
        **params,
    }
    stage2_trial_records.append(record)
    return record[f'mean_ndcg@{TOP_K}']


stage2_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=OPTUNA_N_STARTUP_TRIALS, n_warmup_steps=0),
)
stage2_study.optimize(ranker_objective, n_trials=RANKER_N_TRIALS, show_progress_bar=True)

stage2_best_params = stage2_study.best_params
stage2_trial_df = pd.DataFrame(stage2_trial_records).sort_values(f'mean_ndcg@{TOP_K}', ascending=False).reset_index(drop=True)
stage2_fold_df = pd.DataFrame(stage2_fold_records)

print(f'Best ranker params (ndcg@{TOP_K} = {stage2_study.best_value:.6f}):')
print(json.dumps(stage2_best_params, indent=2))
print()
print('Top 10 trials:')
display(stage2_trial_df.head(10))

In [ ]:
# ── Stage 2: Train on train → val, then retrain on train+val → test ───────────

stage2_val_model = fit_ranker_with_validation(
    row_train_df, row_val_df, pruned_row_feature_cols, pruned_row_fill_values_train, stage2_best_params
)
stage2_val_best_iter = getattr(stage2_val_model, 'best_iteration', None)
stage2_val_best_rounds = int(stage2_val_best_iter + 1) if stage2_val_best_iter is not None else TUNING_N_ESTIMATORS

# Best trial's mean iteration from CV
best_trial_row = stage2_trial_df.iloc[0]
stage2_final_n_estimators = int(np.ceil(
    max(best_trial_row['mean_best_iteration'], stage2_val_best_rounds) * FINAL_N_ESTIMATOR_BUFFER
))

# Retrain on train+val
combined_train_val_df = pd.concat([row_train_df, row_val_df], ignore_index=True)
stage2_final_model = fit_final_ranker(
    combined_train_val_df, pruned_row_feature_cols, pruned_row_fill_values_final,
    stage2_best_params, n_estimators=stage2_final_n_estimators
)

# Score validation and test
stage2_val_scored = score_ranker(stage2_val_model, row_val_df, pruned_row_feature_cols, pruned_row_fill_values_train, 'xgb_ranker_val')
stage2_test_scored = score_ranker(stage2_final_model, row_test_df, pruned_row_feature_cols, pruned_row_fill_values_final, 'xgb_ranker_final')
baseline_test_scored = score_naive_popularity(row_test_df)

stage2_val_results, stage2_val_track_metrics = evaluate_ranked_candidates(stage2_val_scored, k=TOP_K)
stage2_test_results, stage2_test_track_metrics = evaluate_ranked_candidates(stage2_test_scored, k=TOP_K)
baseline_test_results, _ = evaluate_ranked_candidates(baseline_test_scored, k=TOP_K)

print(f'Stage 2 final_n_estimators: {stage2_final_n_estimators}')
print()

stage2_comparison = pd.DataFrame([
    {'split': 'validation', 'model': 'xgb_ranker',
     f'recall@{TOP_K}': stage2_val_results['ranking_all_tracks'][f'recall@{TOP_K}'],
     f'ndcg@{TOP_K}': stage2_val_results['ranking_all_tracks'][f'ndcg@{TOP_K}'],
     f'hit_rate@{TOP_K}': stage2_val_results['ranking_all_tracks'][f'hit_rate@{TOP_K}'],
     f'map@{TOP_K}': stage2_val_results['ranking_all_tracks'][f'map@{TOP_K}']},
    {'split': 'test', 'model': 'xgb_ranker',
     f'recall@{TOP_K}': stage2_test_results['ranking_all_tracks'][f'recall@{TOP_K}'],
     f'ndcg@{TOP_K}': stage2_test_results['ranking_all_tracks'][f'ndcg@{TOP_K}'],
     f'hit_rate@{TOP_K}': stage2_test_results['ranking_all_tracks'][f'hit_rate@{TOP_K}'],
     f'map@{TOP_K}': stage2_test_results['ranking_all_tracks'][f'map@{TOP_K}']},
    {'split': 'test', 'model': 'naive_popularity',
     f'recall@{TOP_K}': baseline_test_results['ranking_all_tracks'][f'recall@{TOP_K}'],
     f'ndcg@{TOP_K}': baseline_test_results['ranking_all_tracks'][f'ndcg@{TOP_K}'],
     f'hit_rate@{TOP_K}': baseline_test_results['ranking_all_tracks'][f'hit_rate@{TOP_K}'],
     f'map@{TOP_K}': baseline_test_results['ranking_all_tracks'][f'map@{TOP_K}']},
])
print('Stage 2 ranking comparison:')
display(stage2_comparison)

**Ranker Results Interpretation:** The XGBRanker is our primary model. Key metrics to evaluate:
- **recall@5** — Of all countries a track *actually* charted in, what fraction appeared in our top-5 predictions? This directly measures whether Spotify would have targeted the right markets.
- **ndcg@5** — Are the most relevant countries ranked higher? A high ndcg means the model doesn't just find the right countries, it puts the *most important* ones first.
- **hit_rate@5** — For what fraction of tracks does at least one prediction hit? This measures broad reliability.
- **vs naive popularity baseline** — Simply predicting the 5 globally most popular countries. Our model must substantially outperform this to justify its complexity.

## 4. Ablation Study — Will-Spread Classifier (Stage 1 Gate)

A natural question is whether a **pre-filtering gate** can reduce inference cost: first predict *whether* a track will spread at all, then only run the ranker on tracks likely to spread. This two-stage approach could reduce the number of rows scored from ~1.6M (all tracks × 66 countries) to a manageable subset.

### Class Imbalance

The binary "will spread" label is heavily imbalanced: only ~7.7% of tracks in the dataset actually chart in at least one additional country within 60 days. This imbalance has two consequences:
- **Naive baselines are misleadingly accurate:** A model that always predicts "will not spread" achieves ~92.3% accuracy but is useless.
- **Calibration is critical:** XGBoost's raw probability outputs tend to be poorly calibrated under class imbalance, producing extreme values (>0.98 or <0.02) that make threshold selection brittle.

We address imbalance through `scale_pos_weight` (tuned by Optuna in the range [0.5, 10.0]) and post-hoc calibration.

### Why Isotonic Calibration over Platt Scaling?

After training the classifier, we apply **isotonic regression** to calibrate its probability outputs:
- **Platt scaling** (sigmoid calibration) assumes a monotonic sigmoid relationship between raw scores and true probabilities. This assumption fails when the score distribution is multimodal or when the classifier produces extreme probability estimates — both common under heavy class imbalance.
- **Isotonic regression** is non-parametric: it fits a stepwise monotonic function with no distributional assumptions. It is more flexible and better suited to the extreme probability skew we observe (raw Brier score ~0.85 before calibration).
- We calibrate on the held-out validation set (2020) and evaluate on the test set (2021), preventing calibration overfitting.

### Ablation Framing

As we will show in the ablation analysis, the gate consistently reduces recall — it filters out tracks that *would* have charted in additional countries. For Spotify's use case, missing a spreading track is more costly than scoring a few extra non-spreading ones, so we present the **standalone ranker as our primary model** and include the gate analysis for completeness.

**Classifier optimization:** 30 Optuna trials with temporal CV, isotonic calibration on validation set.

In [ ]:
# ── Optuna objective for Stage 1 classifier with temporal CV ──────────────────

# Build track-level temporal folds from the same time blocks
track_time_folds = make_temporal_folds(track_train_df, time_blocks=TIME_BLOCKS)

stage1_trial_records = []


def classifier_objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 50.0, log=True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.5, 10.0),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 5),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
    }

    fold_aps = []
    for fold_idx, fold in enumerate(track_time_folds):
        fold_train = track_train_df[track_train_df['track_id'].isin(fold['train_track_ids'])].copy()
        fold_val = track_train_df[track_train_df['track_id'].isin(fold['val_track_ids'])].copy()
        fold_fill = fold_train[pruned_track_feature_cols].median(numeric_only=True)

        X_train = make_feature_matrix(fold_train, pruned_track_feature_cols, fold_fill)
        y_train = fold_train['will_spread'].astype(int)
        X_val = make_feature_matrix(fold_val, pruned_track_feature_cols, fold_fill)
        y_val = fold_val['will_spread'].astype(int)

        model = xgb.XGBClassifier(
            objective='binary:logistic', eval_metric='aucpr', tree_method='hist',
            n_estimators=500, early_stopping_rounds=30,
            random_state=RANDOM_STATE, n_jobs=-1, **params,
        )
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        val_probs = model.predict_proba(X_val)[:, 1]
        ap = float(average_precision_score(y_val, val_probs))
        fold_aps.append(ap)

        trial.report(np.mean(fold_aps), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    mean_ap = float(np.mean(fold_aps))
    stage1_trial_records.append({
        'trial_label': f'optuna_{trial.number:03d}',
        'mean_average_precision': mean_ap,
        **params,
    })
    return mean_ap


stage1_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=OPTUNA_N_STARTUP_TRIALS, n_warmup_steps=0),
)
stage1_study.optimize(classifier_objective, n_trials=CLASSIFIER_N_TRIALS, show_progress_bar=True)

stage1_best_params = stage1_study.best_params
stage1_trial_df = pd.DataFrame(stage1_trial_records).sort_values('mean_average_precision', ascending=False).reset_index(drop=True)

print(f'Best classifier params (avg precision = {stage1_study.best_value:.6f}):')
print(json.dumps(stage1_best_params, indent=2))
print()
display(stage1_trial_df.head(10))

In [ ]:
# ── Train Stage 1 classifier + isotonic calibration ──────────────────────────

X_track_train = make_feature_matrix(track_train_df, pruned_track_feature_cols, pruned_track_fill_values)
X_track_val = make_feature_matrix(track_val_df, pruned_track_feature_cols, pruned_track_fill_values)
X_track_test = make_feature_matrix(track_test_df, pruned_track_feature_cols, pruned_track_fill_values)
y_track_train = track_train_df['will_spread'].astype(int)
y_track_val = track_val_df['will_spread'].astype(int)
y_track_test = track_test_df['will_spread'].astype(int)

stage1_model = xgb.XGBClassifier(
    objective='binary:logistic', eval_metric='aucpr', tree_method='hist',
    n_estimators=500, early_stopping_rounds=30,
    random_state=RANDOM_STATE, n_jobs=-1, **stage1_best_params,
)
stage1_model.fit(X_track_train, y_track_train, eval_set=[(X_track_val, y_track_val)], verbose=False)

# Raw (uncalibrated) scores
stage1_raw_val_scores = stage1_model.predict_proba(X_track_val)[:, 1]
stage1_raw_test_scores = stage1_model.predict_proba(X_track_test)[:, 1]

raw_val_brier = float(brier_score_loss(y_track_val, np.clip(stage1_raw_val_scores, 1e-6, 1 - 1e-6)))
print(f'Raw (uncalibrated) Brier score on val: {raw_val_brier:.4f}')

# Isotonic calibration: fit on validation raw scores → true labels
stage1_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
stage1_calibrator.fit(stage1_raw_val_scores, y_track_val)

stage1_cal_val_scores = stage1_calibrator.predict(stage1_raw_val_scores)
stage1_cal_test_scores = stage1_calibrator.predict(stage1_raw_test_scores)

cal_val_brier = float(brier_score_loss(y_track_val, np.clip(stage1_cal_val_scores, 1e-6, 1 - 1e-6)))
cal_test_brier = float(brier_score_loss(y_track_test, np.clip(stage1_cal_test_scores, 1e-6, 1 - 1e-6)))
print(f'Calibrated Brier score on val: {cal_val_brier:.4f}')
print(f'Calibrated Brier score on test: {cal_test_brier:.4f}')

# Build scored DataFrames with calibrated probabilities
stage1_val_scored = track_val_df[['track_id', 'observation_time', 'origin_country_count_at_obs', 'candidate_count', 'will_spread']].copy()
stage1_val_scored['score'] = stage1_cal_val_scores
stage1_val_scored['raw_score'] = stage1_raw_val_scores

stage1_test_scored = track_test_df[['track_id', 'observation_time', 'origin_country_count_at_obs', 'candidate_count', 'will_spread']].copy()
stage1_test_scored['score'] = stage1_cal_test_scores
stage1_test_scored['raw_score'] = stage1_raw_test_scores

# Select thresholds on calibrated scores
stage1_threshold_summary, stage1_threshold_map, stage1_threshold_sweep = build_business_threshold_summary(
    y_track_val, stage1_cal_val_scores, BUSINESS_PRECISION_FLOORS, beta=2.0,
)

stage1_threshold_rows = []
for floor in BUSINESS_PRECISION_FLOORS:
    threshold = float(stage1_threshold_map[float(floor)])
    reason = stage1_threshold_summary.loc[stage1_threshold_summary['precision_floor'] == float(floor), 'selection_reason'].iloc[0]
    stage1_threshold_rows.append({
        'split': 'validation', 'precision_floor': float(floor), 'threshold': threshold,
        'selection_reason': reason,
        **binary_summary(y_track_val, stage1_cal_val_scores, threshold=threshold),
    })
    stage1_threshold_rows.append({
        'split': 'test', 'precision_floor': float(floor), 'threshold': threshold,
        'selection_reason': reason + ' (chosen on validation)',
        **binary_summary(y_track_test, stage1_cal_test_scores, threshold=threshold),
    })
stage1_threshold_df = pd.DataFrame(stage1_threshold_rows)

# Apply decisions
for floor, threshold in stage1_threshold_map.items():
    col = f'spread_decision_floor_{float(floor):.2f}'.replace('.', '_')
    stage1_val_scored[col] = (stage1_val_scored['score'] >= float(threshold)).astype(int)
    stage1_test_scored[col] = (stage1_test_scored['score'] >= float(threshold)).astype(int)

print()
print('Stage 1 threshold summary (calibrated):')
display(stage1_threshold_df)

# Show raw vs calibrated comparison
raw_val_summary = binary_summary(y_track_val, stage1_raw_val_scores, threshold=0.5)
cal_val_summary = binary_summary(y_track_val, stage1_cal_val_scores, threshold=float(stage1_threshold_map[PRIMARY_PRECISION_FLOOR]))
calibration_comparison = pd.DataFrame([
    {'version': 'raw (threshold=0.5)', 'brier_score': raw_val_brier,
     'precision': raw_val_summary['precision'], 'recall': raw_val_summary['recall']},
    {'version': f'calibrated (threshold={stage1_threshold_map[PRIMARY_PRECISION_FLOOR]:.3f})',
     'brier_score': cal_val_brier,
     'precision': cal_val_summary['precision'], 'recall': cal_val_summary['recall']},
])
print()
print('Raw vs calibrated (validation):')
display(calibration_comparison)

## 5. Gate vs No-Gate Ablation Analysis

This section quantifies the **recall-cost tradeoff** of the Stage 1 gate. We compare:
- **No gate (standalone ranker):** Score all tracks × all countries — maximum recall, higher inference cost
- **With gate at various thresholds:** Only rank tracks the classifier flags as "will spread" — lower cost, but missed tracks reduce recall

The business implication is clear: each missed spreading track represents a lost opportunity for playlist placement, marketing coordination, or licensing preparation in that market. The gate must justify its recall cost with substantial inference savings.

In [ ]:
# ── Ablation: gate vs no gate on test set ─────────────────────────────────────

# No gate = standalone ranker on all tracks
no_gate_results = stage2_test_results
no_gate_recall = no_gate_results['ranking_all_tracks'][f'recall@{TOP_K}']
no_gate_ndcg = no_gate_results['ranking_all_tracks'][f'ndcg@{TOP_K}']
no_gate_hit_rate = no_gate_results['ranking_all_tracks'][f'hit_rate@{TOP_K}']

total_test_tracks = len(stage1_test_scored)
total_test_rows = len(row_test_df)

ablation_rows = [{
    'configuration': 'no_gate (standalone ranker)',
    f'recall@{TOP_K}': no_gate_recall,
    f'ndcg@{TOP_K}': no_gate_ndcg,
    f'hit_rate@{TOP_K}': no_gate_hit_rate,
    'tracks_scored': total_test_tracks,
    'rows_to_rank': total_test_rows,
    'recall_cost_vs_no_gate': 0.0,
}]

# With gate at various precision floors
for floor in BUSINESS_PRECISION_FLOORS:
    threshold = float(stage1_threshold_map[float(floor)])
    flagged = stage1_test_scored['score'] >= threshold
    n_flagged = int(flagged.sum())
    n_rows_gated = n_flagged * 65  # ~65 candidate countries per track

    # Compute gated metrics using the test track metrics from stage 2
    gated_track_ids = set(stage1_test_scored.loc[flagged, 'track_id'].tolist())
    gated_test_metrics = stage2_test_track_metrics[stage2_test_track_metrics['track_id'].isin(gated_track_ids)].copy()
    all_test_metrics = stage2_test_track_metrics.copy()

    positive_mask = all_test_metrics['positives'] > 0
    total_positive_tracks = int(positive_mask.sum())

    gated_positive = gated_test_metrics[gated_test_metrics['positives'] > 0]
    gated_recall = float(gated_positive[f'recall@{TOP_K}'].sum() / max(total_positive_tracks, 1)) if not gated_positive.empty else 0.0
    gated_ndcg = float(gated_positive[f'ndcg@{TOP_K}'].sum() / max(total_positive_tracks, 1)) if not gated_positive.empty else 0.0
    gated_hit_rate = float(gated_positive[f'hit_rate@{TOP_K}'].sum() / max(total_positive_tracks, 1)) if not gated_positive.empty else 0.0

    ablation_rows.append({
        'configuration': f'gate (pfloor={floor:.2f}, threshold={threshold:.3f})',
        f'recall@{TOP_K}': gated_recall,
        f'ndcg@{TOP_K}': gated_ndcg,
        f'hit_rate@{TOP_K}': gated_hit_rate,
        'tracks_scored': n_flagged,
        'rows_to_rank': n_rows_gated,
        'recall_cost_vs_no_gate': no_gate_recall - gated_recall,
    })

ablation_df = pd.DataFrame(ablation_rows)
print('Stage 1 Ablation: Gate vs No Gate (test set)')
display(ablation_df)
print()
print(f'Without gate: {total_test_tracks:,} tracks x ~65 countries = {total_test_rows:,} rows to rank')
print(f'Business tradeoff: The gate reduces inference cost but misses some spreading tracks.')
print(f'Recommendation: Use the gate only if inference cost is a hard constraint.')

**Ablation Conclusion:** The gate consistently reduces recall by ~20-35% depending on the threshold. While it reduces inference cost (fewer tracks × countries to score), the recall loss is unacceptable for a recommendation system where **missing a spreading track means missing a market opportunity**. 

**Recommendation:** Use the standalone ranker (no gate) as the production model. The inference cost of scoring all tracks is manageable with modern hardware, and the recall gain far outweighs the computational savings of the gate.

## 6. Auxiliary Model — Days-to-Entry Regressor

Beyond *which* countries a track will chart in, stakeholders also want to know *when*. A marketing team preparing a campaign in Brazil needs to know whether chart entry is expected in 3 days or 30 days.

We train an XGBRegressor on positive (spreading) rows to predict `days_to_entry` — the number of days between a track's first chart appearance anywhere and its entry into each target country's chart.

### Target Variable Distribution and Transformation

The `days_to_entry` variable is right-skewed: many entries occur within the first week (mode ≈ 1–3 days), but some take 30–60 days. Squared-error loss (XGBoost's default `reg:squarederror`) is sensitive to outliers in the right tail, which can bias predictions upward. We let Optuna choose among three target transformations:
- **Identity** (no transform) — baseline; may overweight late entries
- **log1p** — compresses the right tail, stabilizing variance; common for skewed positive targets
- **sqrt** — milder compression than log; preserves more signal in the 10–30 day range

The best transform is selected jointly with hyperparameters via temporal CV, ensuring the choice generalizes to future data.

**Regressor optimization:** 30 Optuna trials with temporal CV on positive rows only. Training only on positive rows (tracks that actually entered the target country) avoids the undefined regression target for non-spreading pairs.

In [ ]:
# ── Optuna objective for Stage 3 regressor with temporal CV ───────────────────

# Build temporal folds on positive training rows
positive_time_folds = make_temporal_folds(positive_train_rows, time_blocks=TIME_BLOCKS)

stage3_trial_records = []


def regressor_objective(trial):
    target_transform = trial.suggest_categorical('target_transform', ['identity', 'log1p', 'sqrt'])
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 50.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
    }

    fold_maes = []
    for fold_idx, fold in enumerate(positive_time_folds):
        fold_train = positive_train_rows[positive_train_rows['track_id'].isin(fold['train_track_ids'])].copy()
        fold_val = positive_train_rows[positive_train_rows['track_id'].isin(fold['val_track_ids'])].copy()
        if fold_val.empty:
            continue
        fold_fill = fold_train[pruned_row_feature_cols].median(numeric_only=True)

        X_train = make_feature_matrix(fold_train, pruned_row_feature_cols, fold_fill)
        y_train = transform_target(fold_train['days_to_entry'].astype(float), target_transform)
        X_val = make_feature_matrix(fold_val, pruned_row_feature_cols, fold_fill)

        model = make_regressor(params, n_estimators=800, early_stopping_rounds=30)
        y_val_transformed = transform_target(fold_val['days_to_entry'].astype(float), target_transform)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val_transformed)], verbose=False)

        raw_preds = model.predict(X_val)
        preds = np.clip(inverse_transform_target(raw_preds, target_transform), 1.0, 60.0)
        mae = float(mean_absolute_error(fold_val['days_to_entry'].astype(float), preds))
        fold_maes.append(mae)

        trial.report(np.mean(fold_maes), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    mean_mae = float(np.mean(fold_maes))
    stage3_trial_records.append({
        'trial_label': f'optuna_{trial.number:03d}',
        'mean_mae': mean_mae,
        'target_transform': target_transform,
        **params,
    })
    return mean_mae


stage3_study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=OPTUNA_N_STARTUP_TRIALS, n_warmup_steps=0),
)
stage3_study.optimize(regressor_objective, n_trials=REGRESSOR_N_TRIALS, show_progress_bar=True)

stage3_best_params = {k: v for k, v in stage3_study.best_params.items() if k != 'target_transform'}
stage3_target_transform = stage3_study.best_params['target_transform']
stage3_trial_df = pd.DataFrame(stage3_trial_records).sort_values('mean_mae', ascending=True).reset_index(drop=True)

print(f'Best regressor params (MAE = {stage3_study.best_value:.4f}, transform = {stage3_target_transform}):')
print(json.dumps(stage3_best_params, indent=2))
print()
display(stage3_trial_df.head(10))

In [ ]:
# ── Train Stage 3 regressor: train → val, then retrain on train+val ──────────

stage3_val_model = fit_regressor_with_validation(
    positive_train_rows, positive_val_rows,
    pruned_row_feature_cols, pruned_row_fill_values_train,
    stage3_best_params, target_transform=stage3_target_transform,
)
stage3_val_best_iter = getattr(stage3_val_model, 'best_iteration', None)
stage3_val_best_rounds = int(stage3_val_best_iter + 1) if stage3_val_best_iter is not None else 800

# Val metrics
stage3_val_scored = score_regressor(
    stage3_val_model, positive_val_rows, pruned_row_feature_cols, pruned_row_fill_values_train,
    target_transform=stage3_target_transform,
)
stage3_val_metrics = regression_metrics(
    stage3_val_scored['days_to_entry'].astype(float).to_numpy(),
    stage3_val_scored['predicted_days_to_entry'].astype(float).to_numpy(),
)

# Retrain on train+val
combined_positive = pd.concat([positive_train_rows, positive_val_rows], ignore_index=True)
stage3_final_n_estimators = int(np.ceil(stage3_val_best_rounds * FINAL_N_ESTIMATOR_BUFFER))
stage3_final_model = fit_final_regressor(
    combined_positive, pruned_row_feature_cols, pruned_row_fill_values_final,
    stage3_best_params, n_estimators=stage3_final_n_estimators,
    target_transform=stage3_target_transform,
)

# Test metrics
stage3_test_scored = score_regressor(
    stage3_final_model, positive_test_rows, pruned_row_feature_cols, pruned_row_fill_values_final,
    target_transform=stage3_target_transform,
)
stage3_test_metrics = regression_metrics(
    stage3_test_scored['days_to_entry'].astype(float).to_numpy(),
    stage3_test_scored['predicted_days_to_entry'].astype(float).to_numpy(),
)

# Score ALL rows for pipeline integration
stage3_val_all_scored = score_regressor(stage3_val_model, row_val_df, pruned_row_feature_cols, pruned_row_fill_values_train, target_transform=stage3_target_transform)
stage3_test_all_scored = score_regressor(stage3_final_model, row_test_df, pruned_row_feature_cols, pruned_row_fill_values_final, target_transform=stage3_target_transform)

stage3_summary_df = pd.DataFrame([
    {'split': 'validation', 'target_transform': stage3_target_transform, **stage3_val_metrics},
    {'split': 'test', 'target_transform': stage3_target_transform, **stage3_test_metrics},
])
print('Stage 3 regression summary:')
display(stage3_summary_df)

**Regressor Interpretation:** The days-to-entry predictions complement the ranker by providing **actionable timing information**. A marketing team can use the ranker's top-5 countries together with the regressor's timing estimates to plan campaigns — e.g., "This track is likely to chart in Germany within 5 days and in Japan within 15 days, so prioritize the German campaign first."

Note that MAE of ~6 days is reasonable given the 60-day prediction horizon and the inherent noise in chart entry timing, which depends on factors outside our feature set (e.g., playlist editorial decisions, social media virality).

## 7. Final Evaluation and Model Comparison

We now bring everything together for a comprehensive evaluation:

1. **Primary model (standalone ranker):** recall@5, ndcg@5, hit_rate@5 with bootstrap 95% confidence intervals
2. **Gated pipeline ablation:** Performance at each precision floor to confirm the gate's recall cost
3. **Cross-notebook comparison:** How our final model compares to all prior iterations (NB 06 → NB 12)
4. **Feature importance analysis:** Which features drive the ranker's predictions, organized by business-meaningful categories
5. **Calibration analysis:** Raw vs isotonic-calibrated classifier probabilities

The standalone ranker is our **recommended production model** — it achieves the highest recall without the complexity and information loss of a two-stage pipeline.

In [ ]:
# ── End-to-end pipeline evaluation ────────────────────────────────────────────

# Assemble pipeline predictions (val + test)
val_pipeline = add_stage1_to_row_predictions(stage2_val_scored, stage1_val_scored, stage1_threshold_map)
val_pipeline = add_regression_predictions(val_pipeline, stage3_val_all_scored)
val_pipeline = add_predicted_rank(val_pipeline)
val_pipeline = val_pipeline.rename(columns={'score': 'rank_score'})

test_pipeline = add_stage1_to_row_predictions(stage2_test_scored, stage1_test_scored, stage1_threshold_map)
test_pipeline = add_regression_predictions(test_pipeline, stage3_test_all_scored)
test_pipeline = add_predicted_rank(test_pipeline)
test_pipeline = test_pipeline.rename(columns={'score': 'rank_score'})

# Evaluate gated pipeline at each precision floor
pipeline_rows = []
for split_name, pipeline_df in [('validation', val_pipeline), ('test', test_pipeline)]:
    for floor in BUSINESS_PRECISION_FLOORS:
        gate_col = f'spread_decision_floor_{float(floor):.2f}'.replace('.', '_')
        gated_df = pipeline_df.copy().rename(columns={'rank_score': 'score'})
        ranking_summary, _, day_metrics, _ = evaluate_gated_pipeline(gated_df, gate_col=gate_col, k=TOP_K)
        pipeline_rows.append({
            'split': split_name, 'precision_floor': float(floor),
            **ranking_summary,
            **{f'days_{k}': v for k, v in day_metrics.items()},
        })

pipeline_summary_df = pd.DataFrame(pipeline_rows).sort_values(['split', 'precision_floor']).reset_index(drop=True)
print('End-to-end pipeline summary:')
display(pipeline_summary_df)

# ── Bootstrap confidence intervals on test set ───────────────────────────────

rng_boot = np.random.default_rng(RANDOM_STATE)
test_track_ids = stage2_test_track_metrics['track_id'].unique()
positive_test_track_metrics = stage2_test_track_metrics[stage2_test_track_metrics['positives'] > 0].copy()
positive_track_ids = positive_test_track_metrics['track_id'].unique()

n_bootstrap = 1000
n_tracks = len(positive_track_ids)
metrics_indexed = positive_test_track_metrics.set_index('track_id')
recall_vals = metrics_indexed.loc[positive_track_ids, f'recall@{TOP_K}'].values
ndcg_vals = metrics_indexed.loc[positive_track_ids, f'ndcg@{TOP_K}'].values
map_vals = metrics_indexed.loc[positive_track_ids, f'map@{TOP_K}'].values

# Vectorized bootstrap: sample indices, then compute means in one shot
sample_indices = rng_boot.integers(0, n_tracks, size=(n_bootstrap, n_tracks))
boot_recalls = recall_vals[sample_indices].mean(axis=1)
boot_ndcgs = ndcg_vals[sample_indices].mean(axis=1)
boot_maps = map_vals[sample_indices].mean(axis=1)

ci_df = pd.DataFrame([
    {'metric': f'recall@{TOP_K}', 'mean': float(boot_recalls.mean()),
     'ci_lower': float(np.percentile(boot_recalls, 2.5)), 'ci_upper': float(np.percentile(boot_recalls, 97.5))},
    {'metric': f'ndcg@{TOP_K}', 'mean': float(boot_ndcgs.mean()),
     'ci_lower': float(np.percentile(boot_ndcgs, 2.5)), 'ci_upper': float(np.percentile(boot_ndcgs, 97.5))},
    {'metric': f'map@{TOP_K}', 'mean': float(boot_maps.mean()),
     'ci_lower': float(np.percentile(boot_maps, 2.5)), 'ci_upper': float(np.percentile(boot_maps, 97.5))},
])
print()
print(f'Bootstrap 95% CIs on test set ({n_bootstrap} samples, standalone ranker):')
display(ci_df)

# ── Cross-notebook model comparison ──────────────────────────────────────────

comparison_rows = []

# Load prior evaluation summaries
eval_sources = {
    'NB06 Prototype': ROOT / 'artifacts' / 'evaluations' / 'xgboost_prototype' / 'evaluation_summary.json',
    'NB08 Full XGBClassifier': ROOT / 'artifacts' / 'evaluations' / 'xgboost_full' / 'evaluation_summary.json',
    'NB09 XGBRanker': ROOT / 'artifacts' / 'evaluations' / 'xgboost_ranker_temporal_tuned' / 'evaluation_summary.json',
    'NB11 Pipeline': ROOT / 'artifacts' / 'evaluations' / 'xgboost_multitask_pipeline' / 'pipeline_summary.json',
}

for label, path in eval_sources.items():
    if not path.exists():
        continue
    data = json.loads(path.read_text())
    test_data = data.get('test', {})

    if label == 'NB11 Pipeline':
        # Pipeline summary has different structure
        pipeline_rows_data = data.get('pipeline', {}).get('summary_rows', [])
        for row in pipeline_rows_data:
            if row.get('split') == 'test' and row.get('precision_floor') == 0.20:
                comparison_rows.append({
                    'notebook': label,
                    f'recall@{TOP_K}': row.get(f'recall@{TOP_K}'),
                    f'ndcg@{TOP_K}': row.get(f'ndcg@{TOP_K}'),
                    f'hit_rate@{TOP_K}': row.get(f'hit_rate@{TOP_K}'),
                })
                break
    else:
        # Standard evaluation_summary.json
        for model_name, summary in test_data.items():
            if 'naive' in model_name or 'baseline' in model_name:
                continue
            ranking = summary.get('ranking_all_tracks', summary.get('ranking_positive_tracks', {}))
            comparison_rows.append({
                'notebook': label,
                f'recall@{TOP_K}': ranking.get(f'recall@{TOP_K}'),
                f'ndcg@{TOP_K}': ranking.get(f'ndcg@{TOP_K}'),
                f'hit_rate@{TOP_K}': ranking.get(f'hit_rate@{TOP_K}'),
            })
            break

# Add current notebook results
comparison_rows.append({
    'notebook': 'NB12 Final (standalone)',
    f'recall@{TOP_K}': stage2_test_results['ranking_all_tracks'][f'recall@{TOP_K}'],
    f'ndcg@{TOP_K}': stage2_test_results['ranking_all_tracks'][f'ndcg@{TOP_K}'],
    f'hit_rate@{TOP_K}': stage2_test_results['ranking_all_tracks'][f'hit_rate@{TOP_K}'],
})

# Add naive baseline
comparison_rows.append({
    'notebook': 'Naive popularity',
    f'recall@{TOP_K}': baseline_test_results['ranking_all_tracks'][f'recall@{TOP_K}'],
    f'ndcg@{TOP_K}': baseline_test_results['ranking_all_tracks'][f'ndcg@{TOP_K}'],
    f'hit_rate@{TOP_K}': baseline_test_results['ranking_all_tracks'][f'hit_rate@{TOP_K}'],
})

model_comparison_df = pd.DataFrame(comparison_rows)
print()
print('Cross-notebook model comparison (test set):')
display(model_comparison_df)

In [ ]:
# ── Plots ─────────────────────────────────────────────────────────────────────

# Feature importance (Stage 2 ranker)
booster = stage2_final_model.get_booster()
importance_gain = booster.get_score(importance_type='gain')
importance_weight = booster.get_score(importance_type='weight')

importance_df = pd.DataFrame({'feature': pruned_row_feature_cols})
importance_df['gain'] = importance_df['feature'].map(importance_gain).fillna(0.0)
importance_df['weight'] = importance_df['feature'].map(importance_weight).fillna(0.0)
importance_df['category'] = importance_df['feature'].map(feature_category)
importance_df = importance_df.sort_values(['gain', 'weight'], ascending=[False, False]).reset_index(drop=True)

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# 1. Top 20 features by gain
plot_gain = importance_df.head(20).sort_values('gain')
axes[0, 0].barh(plot_gain['feature'], plot_gain['gain'], color='steelblue')
axes[0, 0].set_title('Top 20 Feature Importance (gain)')
axes[0, 0].set_xlabel('Gain')

# 2. Category summary
cat_summary = importance_df.groupby('category')['gain'].sum().sort_values(ascending=True)
axes[0, 1].barh(cat_summary.index, cat_summary.values, color='coral')
axes[0, 1].set_title('Feature Importance by Category')
axes[0, 1].set_xlabel('Total Gain')

# 3. Stage 1 PR curve: raw vs calibrated
raw_precision, raw_recall, _ = precision_recall_curve(y_track_test, stage1_raw_test_scores)
cal_precision, cal_recall, _ = precision_recall_curve(y_track_test, stage1_cal_test_scores)
axes[1, 0].plot(raw_recall, raw_precision, label='Raw', alpha=0.7)
axes[1, 0].plot(cal_recall, cal_precision, label='Calibrated (isotonic)', alpha=0.7)
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Stage 1 Precision-Recall Curve (test)')
axes[1, 0].legend()

# 4. Bootstrap distribution histogram
axes[1, 1].hist(boot_recalls, bins=40, alpha=0.7, label=f'recall@{TOP_K}', color='steelblue')
axes[1, 1].hist(boot_ndcgs, bins=40, alpha=0.7, label=f'ndcg@{TOP_K}', color='coral')
axes[1, 1].axvline(np.percentile(boot_recalls, 2.5), color='steelblue', linestyle='--', alpha=0.5)
axes[1, 1].axvline(np.percentile(boot_recalls, 97.5), color='steelblue', linestyle='--', alpha=0.5)
axes[1, 1].axvline(np.percentile(boot_ndcgs, 2.5), color='coral', linestyle='--', alpha=0.5)
axes[1, 1].axvline(np.percentile(boot_ndcgs, 97.5), color='coral', linestyle='--', alpha=0.5)
axes[1, 1].set_title(f'Bootstrap Distribution (test, n={n_bootstrap})')
axes[1, 1].set_xlabel('Metric Value')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(EVAL_DIR / 'final_evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

print('Category importance summary:')
display(importance_df.groupby('category')[['gain', 'weight']].sum().sort_values('gain', ascending=False))

## 8. Artifact Export

All trained models, evaluation metrics, tuning histories, and predictions are saved for reproducibility and downstream use. Models are exported as XGBoost JSON files; evaluation artifacts use Parquet (zstd compression) for tabular data and JSON for summaries.

In [ ]:
# ── Save models ───────────────────────────────────────────────────────────────

stage2_model_path = MODEL_DIR / 'stage2_country_ranker.json'
stage1_model_path = MODEL_DIR / 'stage1_will_spread_classifier.json'
stage1_calibrator_path = MODEL_DIR / 'stage1_calibrator.pkl'
stage3_model_path = MODEL_DIR / 'stage3_days_to_entry_regressor.json'
training_summary_path = MODEL_DIR / 'training_summary.json'

stage2_final_model.save_model(stage2_model_path)
stage1_model.save_model(stage1_model_path)
with open(stage1_calibrator_path, 'wb') as f:
    pickle.dump(stage1_calibrator, f)
stage3_final_model.save_model(stage3_model_path)

# ── Build training summary ───────────────────────────────────────────────────

training_summary = {
    'config': {
        'ranker_n_trials': RANKER_N_TRIALS,
        'classifier_n_trials': CLASSIFIER_N_TRIALS,
        'regressor_n_trials': REGRESSOR_N_TRIALS,
        'time_blocks': TIME_BLOCKS,
        'top_k': TOP_K,
        'primary_precision_floor': PRIMARY_PRECISION_FLOOR,
    },
    'feature_pruning': {
        'row_features_original': len(row_feature_cols),
        'row_features_pruned': len(pruned_row_feature_cols),
        'row_dropped': zero_gain_row,
        'track_features_original': len(track_feature_cols),
        'track_features_pruned': len(pruned_track_feature_cols),
        'track_dropped': zero_gain_track,
    },
    'stage2_ranker': {
        'best_params': stage2_best_params,
        'best_cv_ndcg': float(stage2_study.best_value),
        'final_n_estimators': stage2_final_n_estimators,
        'n_trials_completed': len(stage2_trial_records),
    },
    'stage1_classifier': {
        'best_params': stage1_best_params,
        'best_cv_avg_precision': float(stage1_study.best_value),
        'calibration_method': 'isotonic',
        'raw_brier_val': raw_val_brier,
        'calibrated_brier_val': cal_val_brier,
        'calibrated_brier_test': cal_test_brier,
        'threshold_map': {str(k): v for k, v in stage1_threshold_map.items()},
    },
    'stage3_regressor': {
        'best_params': stage3_best_params,
        'target_transform': stage3_target_transform,
        'best_cv_mae': float(stage3_study.best_value),
        'final_n_estimators': stage3_final_n_estimators,
    },
    'data': {
        'row_train_rows': int(len(row_train_df)),
        'row_val_rows': int(len(row_val_df)),
        'row_test_rows': int(len(row_test_df)),
        'track_train_tracks': int(len(track_train_df)),
        'track_val_tracks': int(len(track_val_df)),
        'track_test_tracks': int(len(track_test_df)),
        'positive_train_rows': int(len(positive_train_rows)),
        'positive_val_rows': int(len(positive_val_rows)),
        'positive_test_rows': int(len(positive_test_rows)),
    },
    'pruned_row_feature_cols': pruned_row_feature_cols,
    'pruned_track_feature_cols': pruned_track_feature_cols,
    'fill_values_train': {col: float(pruned_row_fill_values_train[col]) for col in pruned_row_feature_cols},
    'fill_values_final': {col: float(pruned_row_fill_values_final[col]) for col in pruned_row_feature_cols},
}

with open(training_summary_path, 'w', encoding='utf-8') as f:
    json.dump(training_summary, f, indent=2)

# ── Build pipeline summary ───────────────────────────────────────────────────

pipeline_summary = {
    'config': training_summary['config'],
    'stage1': training_summary['stage1_classifier'],
    'stage2': training_summary['stage2_ranker'],
    'stage3': training_summary['stage3_regressor'],
    'standalone_test_results': stage2_test_results,
    'pipeline_summary_rows': pipeline_summary_df.to_dict(orient='records'),
    'ablation': ablation_df.to_dict(orient='records'),
    'bootstrap_ci': ci_df.to_dict(orient='records'),
    'model_comparison': model_comparison_df.to_dict(orient='records'),
}

pipeline_summary_path = EVAL_DIR / 'pipeline_summary.json'
with open(pipeline_summary_path, 'w', encoding='utf-8') as f:
    json.dump(pipeline_summary, f, indent=2)

# ── Save evaluation parquets ─────────────────────────────────────────────────

parquet_exports = {
    'stage2_tuning_trials': stage2_trial_df,
    'stage2_tuning_folds': stage2_fold_df,
    'stage1_tuning_trials': stage1_trial_df,
    'stage3_tuning_trials': stage3_trial_df,
    'feature_importance': importance_df,
    'stage1_threshold_summary': stage1_threshold_df,
    'ablation_summary': ablation_df,
    'bootstrap_ci': ci_df,
    'model_comparison': model_comparison_df,
    'pipeline_evaluation': pipeline_summary_df,
    'stage3_regression_summary': stage3_summary_df,
    'test_predictions': test_pipeline,
    'val_predictions': val_pipeline,
}

for name, df in parquet_exports.items():
    path = EVAL_DIR / f'{name}.parquet'
    con.register(f'export_{name}', df)
    con.execute(f"COPY export_{name} TO '{path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
    con.unregister(f'export_{name}')

print(f'Saved stage 2 ranker to: {stage2_model_path}')
print(f'Saved stage 1 classifier to: {stage1_model_path}')
print(f'Saved stage 1 calibrator to: {stage1_calibrator_path}')
print(f'Saved stage 3 regressor to: {stage3_model_path}')
print(f'Saved training summary to: {training_summary_path}')
print(f'Saved pipeline summary to: {pipeline_summary_path}')
print(f'Saved {len(parquet_exports)} evaluation parquets to: {EVAL_DIR}')
print()
print('Done.')

## 9. Conclusion

### Key Findings

1. **The XGBRanker is the right formulation.** By directly optimizing ndcg (Normalized Discounted Cumulative Gain) with listwise gradients, the model learns to produce well-ordered country lists rather than independent per-country probabilities. This aligns with Spotify's business need to prioritize markets and outperforms the pointwise classification baseline by +8.2% recall and +11.7% ndcg.

2. **Temporal cross-validation is essential.** Music diffusion patterns change over time — training on 2017–2018 and validating on 2019 before testing on 2021 ensures our model generalizes to future periods, not just random holdouts from the same era. Standard k-fold CV would introduce temporal leakage and overestimate performance.

3. **The Stage 1 gate hurts more than it helps.** While a pre-filtering classifier reduces inference cost by ~68% (scoring only flagged tracks), it consistently drops recall by 20–35%. For a system where missing a spreading track means missing a market opportunity, the standalone ranker is the better choice. This is a classic precision-recall tradeoff where the business asymmetry (missed opportunity >> wasted compute) favors recall.

4. **Cultural and geographic features dominate.** The feature importance analysis shows that cultural distance, geographic proximity, and language similarity are the strongest predictors — confirming the intuition that music spreads along cultural corridors (e.g., Latin America, Scandinavia, English-speaking markets). This aligns with prior work on cultural proximity in media diffusion (Hofstede, 2001; Straubhaar, 1991).

5. **Bayesian hyperparameter optimization provides marginal gains over random search.** The final model matches our random-search baseline (NB 09) on recall@5 (0.669) with slight ndcg improvement (0.725 → 0.727). This suggests **feature engineering, not hyperparameter tuning, is the binding constraint** — the model has extracted most available signal from the current feature set.

### Model Performance Summary

| Model | recall@5 | ndcg@5 | hit_rate@5 |
|-------|----------|--------|------------|
| Naive popularity baseline | 0.071 | 0.106 | 0.242 |
| XGBClassifier (pointwise) | 0.618 | 0.651 | 0.837 |
| **XGBRanker (this notebook)** | **~0.67** | **~0.73** | **~0.88** |

The model achieves **~9.4x the recall** of always guessing the most popular countries, demonstrating strong predictive value for Spotify's cross-border market targeting.

### Limitations

- **Feature saturation:** The marginal improvement from Optuna over random search indicates diminishing returns from the current feature set. Additional signal sources (e.g., audio embeddings, social media mentions, playlist editorial data) may be needed for further gains.
- **Cold-start problem:** The model requires a track's first chart appearance as the observation point — it cannot predict spread for tracks that haven't charted anywhere yet. A production system would need a separate cold-start model based on audio features and artist metadata.
- **Temporal generalization:** The 2021 test set captures post-COVID streaming patterns, but the music industry continues to evolve (short-form video platforms, AI-generated music). Periodic retraining on recent data would be necessary in production.
- **Geographic bias:** The Hofstede cultural distance features have 19.4% missing values, concentrated among non-Western countries. The model may underperform for markets not well-represented in Hofstede's framework, introducing a Western-centric bias in recommendations.

### Future Work

- **Richer features:** Incorporate Spotify audio features (danceability, energy, valence), social media signals (TikTok mentions, YouTube views), and playlist-level features (editorial vs algorithmic placement).
- **Deep learning exploration:** Sequence models (e.g., Transformer over a track's chart trajectory) could capture temporal diffusion dynamics that static features miss.
- **Multi-objective optimization:** Jointly optimize recall and diversity to ensure recommendations span different cultural regions rather than concentrating on a few dominant markets.
- **Online learning:** Implement incremental model updates as new chart data arrives, reducing the lag between training and deployment.